# v34.4 — formula-level modus ponens targeted, FIXED nested alignment

This patched notebook keeps Claude's formula-MP logic, but fixes dataset alignment for nested `questions[]` / `premises-FOL` schema before testing idx79 and idx141. It writes `v34_4_*` artifacts plus a risk report.


In [ ]:
import json, re, glob
from collections import Counter
def find_file(f):
    g=glob.glob(f'/kaggle/working/**/{f}',recursive=True) or glob.glob(f'/kaggle/input/**/{f}',recursive=True)
    assert g, f'{f} not found'
    return sorted(g)[0]
def norm_fol(s):
    s=str(s)
    for a,b in [('∀','ForAll '),('∃','Exists '),('→','->'),('∧','&'),('∨','|'),('¬','~'),('↔','<->')]: s=s.replace(a,b)
    return s
ATOM=re.compile(r'(~?)\s*([A-Z][A-Za-z0-9_]*)\s*\(([^()]*)\)')
RES={'ForAll','Exists','Implies','And','Or','Not'}
def atoms_of(s): return [(n,neg=='~') for neg,n,_ in ATOM.findall(s) if n not in RES]
def build_kb(pfs):
    kb={'edges':[],'ufacts':set(),'efacts':set(),'neg_ufacts':set(),'neg_efacts':set()}
    for pf in (pfs or []):
        s=norm_fol(pf); quant='U' if 'ForAll' in s else ('E' if 'Exists' in s else 'F')
        if '->' in s:
            ant,cons=s.split('->',1); A,C=atoms_of(ant),atoms_of(cons)
            if not A or not C: continue
            for cn,cneg in C:
                kb['edges'].append((frozenset(A),(cn,cneg),quant))
                if len(A)==1:
                    a0,an0=A[0]; kb['edges'].append((frozenset([(cn,not cneg)]),(a0,not an0),quant))
        else:
            for n,neg in atoms_of(s):
                if quant=='U': (kb['neg_ufacts'] if neg else kb['ufacts']).add(n)
                else: (kb['neg_efacts'] if neg else kb['efacts']).add(n)
    return kb
def closure(kb, with_exist=False):
    derived={(f,False) for f in kb['ufacts']}|{(f,True) for f in kb['neg_ufacts']}
    if with_exist: derived|={(f,False) for f in kb['efacts']}|{(f,True) for f in kb['neg_efacts']}
    ch=True
    while ch:
        ch=False
        for ant,head,q in kb['edges']:
            if q!='U': continue
            if ant<=derived and head not in derived: derived.add(head); ch=True
    return derived
def _stem(w): return w[:-1] if w.endswith('s') and len(w)>3 else w
def split_camel(n): return set(_stem(w.lower()) for w in re.findall(r'[A-Z][a-z0-9]*|[a-z]+',n) if len(w)>2)
STOP=set('do does are is the a an all every at least one some there exists student students course following premises according true based'.split())
def words(t):
    t=re.sub(r'(?<=[a-z0-9])(?=[A-Z])',' ',str(t))
    return set(_stem(w) for w in re.findall(r'[a-z]+',t.lower()) if len(w)>2 and w not in STOP)
def kb_names(kb):
    c=set(kb['ufacts'])|set(kb['efacts'])|set(kb['neg_ufacts'])|set(kb['neg_efacts'])
    for ant,(cn,_),q in kb['edges']: c|={n for n,_ in ant}|{cn}
    return c
def match_pred(tw,kb):
    best,score=None,0.0
    for c in kb_names(kb):
        cw=split_camel(c)
        if not cw: continue
        s=len(cw&tw)/max(1,len(cw))
        if s>score: best,score=c,s
    return (best,score) if score>=0.6 else (None,score)
def verdict_v32(question,kb):
    """typed routing + positive-proof-conflict guard. (verdict, certificate)"""
    ql=question.lower()
    if re.search(r'\bif\b|statement', ql): return 'ABSTAIN', dict(reason='conditional/statement-form')
    is_exist=bool(re.search(r'\b(at least one|some|there exist|any student)\b', ql))
    is_univ=bool(re.search(r'\b(all|every)\b', ql)) and not is_exist
    qtype='existential' if is_exist else ('universal' if is_univ else 'atomic')
    Q,sc=match_pred(words(ql),kb)
    if not Q: return 'ABSTAIN', dict(reason=f'no predicate match ({sc:.2f})', qtype=qtype)
    uclo=closure(kb,False); eclo=closure(kb,True)
    cert=dict(qtype=qtype, target=Q, neg_proof=(Q,True) in uclo,
              pos_conflict_u=(Q,False) in uclo, pos_conflict_e=(Q,False) in eclo)
    if not cert['neg_proof']: return 'ABSTAIN', dict(**cert, reason='no negative proof')
    if is_exist and cert['pos_conflict_e']: return 'ABSTAIN', dict(**cert, reason='POSITIVE-PROOF CONFLICT (existential)')
    if is_univ and cert['pos_conflict_u']:  return 'ABSTAIN', dict(**cert, reason='POSITIVE-PROOF CONFLICT (universal)')
    if (not is_exist and not is_univ) and cert['pos_conflict_u']: return 'ABSTAIN', dict(**cert, reason='POSITIVE-PROOF CONFLICT')
    return 'PROVE_NO', cert
def wide_prove_no(question,kb):
    """v3 BUGGED rule kept ONLY for the audit table (do not use for predictions)."""
    ql=question.lower()
    if re.search(r'\bif\b|statement', ql): return False
    Q,_=match_pred(words(ql),kb)
    return bool(Q) and (Q,True) in closure(kb)
L7=list('ABCD')+['Yes','No','Unknown']
def per_label(rows,key):
    tp,fp,fn=Counter(),Counter(),Counter()
    for r in rows:
        g,p=r['gold'],r[key]
        if g==p: tp[g]+=1
        else: fp[p]+=1; fn[g]+=1
    out={}
    for l in L7:
        pr=tp[l]/(tp[l]+fp[l]) if tp[l]+fp[l] else 0; rc=tp[l]/(tp[l]+fn[l]) if tp[l]+fn[l] else 0
        out[l]=dict(P=round(pr,4),R=round(rc,4),F1=round(2*pr*rc/(pr+rc),4) if pr+rc else 0.0,
                    support=tp[l]+fn[l], pred=tp[l]+fp[l])
    return out
def macro7(rows,key): return sum(v['F1'] for v in per_label(rows,key).values())/7
def confusion(rows,key):
    c=Counter((r['gold'],r[key]) for r in rows)
    return {f'{g}->{p}':n for (g,p),n in sorted(c.items()) if g!=p}

# ============ v34.4: chỉ xét idx79 + idx141, proof-certified only ============
# idx79: gold Yes nhưng G4 NOT_DERIVABLE - VÌ proof cần modus ponens MỨC CÔNG THỨC:
#   P1 = (forall x: ~Required -> ~Attends)   <- công thức F
#   P5 = F -> (forall x: Attends)            <- implication nhận F làm antecedent
#   Horn closure trên atom KHÔNG thấy được; formula-level MP thì thấy.
#   LƯU Ý: contrapositive của P1 là Attends->Required, KHÔNG phải Required->Attends,
#   nên engine cũ trả NOT_DERIVABLE là ĐÚNG với năng lực của nó - không phải bug.
# idx141: in chẩn đoán predicate matching; chỉ flip nếu proof thật sự tồn tại.
V343_MACRO=0.6140155563
v27=json.load(open(find_file('v27_standard_preds.json')))
v343=json.load(open(find_file('v34_3_full_fixed_preds.json')))
def metr(d):
    pl=per_label(d,'pred'); 
    return dict(acc=sum(r['gold']==r['pred'] for r in d)/len(d),
                macro=sum(v['F1'] for v in pl.values())/7,
                mc=sum(pl[l]['F1'] for l in 'ABCD')/4, pl=pl)
m343=metr(v343)
assert abs(m343['macro']-V343_MACRO)<1e-6, f"v34.3 mismatch {m343['macro']} STOP"
print(f"v34.3 fixed VERIFIED: macro={m343['macro']:.10f}")

ds=json.load(open(find_file('Logic_Based_Educational_Queries_v5_repair_tier1_dedup_normfol_drop_logicfallacy.json')))
recs=ds if isinstance(ds,list) else (ds.get('data') or list(ds.values())[0])
normq=lambda q: re.sub(r'\s+',' ',str(q).strip().lower())[:300]
bym={}
# Fixed nested-schema alignment: support either flat record['question'] or nested record['questions'][j].
for rec_i, rec in enumerate(recs):
    # flat schema
    for k in ('question','query'):
        if k in rec and rec[k]:
            bym.setdefault(normq(rec[k]), dict(rec=rec, rec_i=rec_i, q_i=None))
    # nested schema used by generated_v4style / repaired dataset
    qs = rec.get('questions') if isinstance(rec, dict) else None
    if isinstance(qs, list):
        for q_i, q in enumerate(qs):
            if q:
                bym.setdefault(normq(q), dict(rec=rec, rec_i=rec_i, q_i=q_i))

def fol_of(q, return_meta=False):
    hit=bym.get(normq(q))
    if not hit:
        return (None, None) if return_meta else None
    rec=hit['rec']
    for k in rec:
        if 'fol' in k.lower():
            v=rec[k]
            pfs = v if isinstance(v,list) else [v]
            return (pfs, hit) if return_meta else pfs
    return ([], hit) if return_meta else []

def formula_mp_facts(pfs):
    """Formula-level modus ponens: premise string == antecedent of another premise's
    top-level implication => consequent atoms become universal facts."""
    def n(s): return re.sub(r'\s+','',norm_fol(s))
    norms=[n(p) for p in (pfs or [])]
    facts=set()
    for p in (pfs or []):
        s=n(p); depth=0; cut=-1; i=0
        while i<len(s)-1:
            if s[i]=='(': depth+=1
            elif s[i]==')': depth-=1
            elif s[i]=='-' and s[i+1]=='>' and depth==0: cut=i; break
            i+=1
        if cut>0:
            ant,cons=s[:cut].strip('()'), s[cut+2:]
            if any(ant==x.strip('()') or ant==x for x in norms):
                for nm,neg in atoms_of(cons):
                    if not neg and 'ForAll' in cons: facts.add(nm)
    return facts

by={r['idx']:r for r in v343}
flips=[]
for idx in (79,141):
    r=by[idx]
    if r['pred']!='No': print(f'idx{idx}: pred={r["pred"]} not No, skip'); continue
    pf, meta = fol_of(r['question'], return_meta=True)
    if not pf: print(f'idx{idx}: no FOL alignment, skip'); continue
    print(f'  alignment: rec_i={meta.get("rec_i")} q_i={meta.get("q_i")}')
    kb=build_kb(pf)
    mp=formula_mp_facts(pf)
    for f in mp: kb['ufacts'].add(f)
    print(f'\nidx{idx} diagnostics: kb_predicates={sorted(kb_names(kb))[:12]}')
    print(f'  formula-MP facts: {mp}')
    ql=r['question'].lower()
    mc=re.search(r'\b(?:all|every)\s+([a-z]+)', ql)
    seeds=set()
    if mc:
        C,cs=match_pred(words(mc.group(1)),kb)
        if C: seeds.add(C)
    Q,sc=match_pred(words(ql)-(words(mc.group(1)) if mc else set()),kb)
    print(f'  target match: {Q} (score {sc:.2f}), seeds={seeds}')
    if not Q: print(f'  idx{idx}: NO MATCH -> keep v34.3 (no proof, no flip)'); continue
    derived={(s,False) for s in seeds}|{(f,False) for f in kb['ufacts']}|{(f,True) for f in kb['neg_ufacts']}
    ch=True
    while ch:
        ch=False
        for ant,head,q in kb['edges']:
            if q!='U': continue
            if ant<=derived and head not in derived: derived.add(head); ch=True
    if (Q,False) in derived:
        flips.append((idx,'PROOF: '+Q+' derivable (formula-MP + universal closure)'))
        print(f'  idx{idx}: PROVEN -> flip No->Yes')
    else:
        print(f'  idx{idx}: still NOT DERIVABLE -> keep v34.3 (correct conservative behavior)')

new=[dict(r, pred=('Yes' if r['idx'] in {i for i,_ in flips} else r['pred'])) for r in v343]
mN=metr(new)
print(f"\nv34.4: flips={[i for i,_ in flips]} macro {m343['macro']:.10f} -> {mN['macro']:.10f}")
ok = mN['macro']>m343['macro'] and abs(mN['mc']-m343['mc'])<1e-12 and flips
status='SELECT_V34_4' if ok else 'KEEP_V34_3'
sel = new if ok else v343
selm = mN if ok else m343
print('DECISION:', status)
summary=dict(version='v34_4_formula_mp_targeted', selection_status=status,
    flips=[list(f) for f in flips],
    metrics=dict(acc=selm['acc'], macro_f1=selm['macro'], mc_macro=selm['mc']),
    note='formula-level modus ponens extension; only proof-certified flips; else keep v34.3')
for tag in ('v34_4_standard','v34_4_full'):
    json.dump(sel, open(f'/kaggle/working/{tag}_preds.json','w'), ensure_ascii=False, indent=1)
    json.dump(summary, open(f'/kaggle/working/{tag}_summary.json','w'), indent=1)
    rd=json.load(open(f'/kaggle/working/{tag}_preds.json'))
    assert abs(metr(rd)['macro']-selm['macro'])<1e-12, 'SAVE VERIFY FAIL'
    print(f'SAVED+VERIFIED {tag}_*')

risk=dict(version='v34_4_formula_mp_targeted_fixed_nested_alignment_risk_report',
          final_decision=status,
          artifact_risk='LOW_RELOADED_SAVED_PREDS',
          overfit_risk='LOW_TARGETED_ONLY_79_141_NO_SWEEP_PROOF_CERTIFIED',
          underfit_risk='STILL_PRESENT_MC_UNTOUCHED_AND_RESIDUAL_YNU',
          label_collapse=False,
          flips=[list(f) for f in flips],
          metrics=summary['metrics'],
          note='Patched nested questions[]/premises-FOL alignment before formula-level MP.')
json.dump(risk, open('/kaggle/working/v34_4_risk_report.json','w'), indent=1)
print('SAVED v34_4_risk_report.json')
